In [0]:
accounts_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/accounts.csv", header=True, inferSchema=True)
products_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/products.csv", header=True, inferSchema=True)
sales_pipeline_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/sales_pipeline.csv", header=True, inferSchema=True)
sales_teams_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/sales_teams.csv", header=True, inferSchema=True)
data_dictionary_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/data_dictionary.csv", header=True, inferSchema=True)

In [0]:
accounts_df.show(5)

+----------------+---------+----------------+-------+---------+---------------+-------------+
|         account|   sector|year_established|revenue|employees|office_location|subsidiary_of|
+----------------+---------+----------------+-------+---------+---------------+-------------+
|Acme Corporation|technolgy|            1996|1100.04|     2822|  United States|         NULL|
|      Betasoloin|  medical|            1999| 251.41|      495|  United States|         NULL|
|        Betatech|  medical|            1986| 647.18|     1185|          Kenya|         NULL|
|      Bioholding|  medical|            2012| 587.34|     1356|     Philipines|         NULL|
|         Bioplex|  medical|            1991| 326.82|     1016|  United States|         NULL|
+----------------+---------+----------------+-------+---------+---------------+-------------+
only showing top 5 rows


In [0]:
print(accounts_df.columns)
print(data_dictionary_df.columns)
print(products_df.columns)
print(sales_pipeline_df.columns)
print(sales_teams_df.columns)

['account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of']
['Table', 'Field', 'Description']
['product', 'series', 'sales_price']
['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']
['sales_agent', 'manager', 'regional_office']


In [0]:
accounts_df = accounts_df.withColumnRenamed("subsidiary_of","parent_company")
data_dictionary_df = data_dictionary_df.withColumnRenamed("Table","table").withColumnRenamed("Field","field").withColumnRenamed("Description","description")

accounts_df.show(2)
data_dictionary_df.show(2)

+----------------+---------+----------------+-------+---------+---------------+--------------+
|         account|   sector|year_established|revenue|employees|office_location|parent_company|
+----------------+---------+----------------+-------+---------+---------------+--------------+
|Acme Corporation|technolgy|            1996|1100.04|     2822|  United States|          NULL|
|      Betasoloin|  medical|            1999| 251.41|      495|  United States|          NULL|
+----------------+---------+----------------+-------+---------+---------------+--------------+
only showing top 2 rows
+--------+-------+------------+
|   table|  field| description|
+--------+-------+------------+
|accounts|account|Company name|
|accounts| sector|    Industry|
+--------+-------+------------+
only showing top 2 rows


In [0]:
from pyspark.sql.functions import col, when, sum

In [0]:
accounts_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/accounts.csv", header=True, inferSchema=True)
products_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/products.csv", header=True, inferSchema=True)
sales_pipeline_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/sales_pipeline.csv", header=True, inferSchema=True)
sales_teams_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/sales_teams.csv", header=True, inferSchema=True)
data_dictionary_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/data_dictionary.csv", header=True, inferSchema=True)

accounts_df = accounts_df.fillna({"subsidiary_of": "Independent"})
sales_pipeline_df = sales_pipeline_df.fillna({"account": "unknown"})

print(accounts_df.columns)
accounts_df.filter(col("subsidiary_of") == "Independent").count()

# display(null_counts_accounts_df)
# display(null_counts_data_dictionary_df)
# display(null_counts_products_df)
# display(null_counts_sales_pipeline_df)
# display(null_counts_sales_teams_df)

['account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of']


70

In [0]:
null_counts_accounts_df = accounts_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in accounts_df.columns])
null_counts_data_dictionary_df = data_dictionary_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in data_dictionary_df.columns])
null_counts_products_df = products_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in products_df.columns])
null_counts_sales_pipeline_df = sales_pipeline_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in sales_pipeline_df.columns])
null_counts_sales_teams_df = sales_teams_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in sales_teams_df.columns])


In [0]:
null_counts_accounts_df.display()

account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,0,0,0,0,0,0


In [0]:
accounts_df.show(5)

+----------------+---------+----------------+-------+---------+---------------+--------------+
|         account|   sector|year_established|revenue|employees|office_location|parent_company|
+----------------+---------+----------------+-------+---------+---------------+--------------+
|Acme Corporation|technolgy|            1996|1100.04|     2822|  United States|   Independent|
|      Betasoloin|  medical|            1999| 251.41|      495|  United States|   Independent|
|        Betatech|  medical|            1986| 647.18|     1185|          Kenya|   Independent|
|      Bioholding|  medical|            2012| 587.34|     1356|     Philipines|   Independent|
|         Bioplex|  medical|            1991| 326.82|     1016|  United States|   Independent|
+----------------+---------+----------------+-------+---------+---------------+--------------+
only showing top 5 rows


In [0]:
sales_pipeline_df.show(5)

+--------------+---------------+--------------+-------+----------+-----------+----------+-----------+
|opportunity_id|    sales_agent|       product|account|deal_stage|engage_date|close_date|close_value|
+--------------+---------------+--------------+-------+----------+-----------+----------+-----------+
|      1C1I7A6R|    Moses Frase|GTX Plus Basic|Cancity|       Won| 2016-10-20|2017-03-01|       1054|
|      Z063OYW0|Darcel Schlecht|        GTXPro|  Isdom|       Won| 2016-10-25|2017-03-11|       4514|
|      EC4QE1BX|Darcel Schlecht|    MG Special|Cancity|       Won| 2016-10-25|2017-03-07|         50|
|      MV1LWRNH|    Moses Frase|     GTX Basic|Codehow|       Won| 2016-10-25|2017-03-09|        588|
|      PE84CX4O|      Zane Levy|     GTX Basic| Hatfan|       Won| 2016-10-25|2017-03-02|        517|
+--------------+---------------+--------------+-------+----------+-----------+----------+-----------+
only showing top 5 rows


In [0]:
null_counts_accounts_df.display()

account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,0,0,0,0,0,0


In [0]:
accounts_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/accounts.csv")
sales_pipeline_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/accounts.csv")
sales_teams_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/accounts.csv")
products_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/accounts.csv")
data_dictionary_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/accounts.csv")